# Highway Lane Analytics - recruiter demo

Run the full vision pipeline on a 55-second real clip from the KM 58+400 camera
(Tj. Priok, 1024x768). You will get an annotated video, lane events, lane changes,
and hogging indications in a few minutes, even on CPU.

- **Dashboard:** https://lane.syaifudin-fam.my.id
- **Repository:** https://github.com/ibrahim119920/highway-lane-analytics
- **Guide:** `docs/DEMO_FOR_RECRUITERS.md`

This notebook works in **Colab** and **Kaggle**. The demo runs locally by default;
uploading to the dashboard is optional and needs a token.


## 1. Clone, install, register `src`

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/ibrahim119920/highway-lane-analytics.git"
PROJECT = "highway-lane-occupancy"

if not Path(PROJECT).is_dir():
    subprocess.run(["git", "clone", REPO, PROJECT], check=True)
os.chdir(PROJECT)
subprocess.run(["git", "pull", "--ff-only"], check=False)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"],
    check=True,
)

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import ultralytics
import torch
from lane_analytics.config import load_config
from lane_analytics.pipeline import process_video

DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("repo        :", ROOT)
print("ultralytics :", ultralytics.__version__)
print("device      :", DEVICE)


## 2. Settings

- `REPLAY = False` runs the pipeline only (safe for any visitor).
- `REPLAY = True` also uploads the run so it appears on the dashboard. The notebook
  owner must put the pipeline token in `PIPELINE_TOKEN` below before publishing.
- `VIDEO_URL` points at the 5-second sample committed in this repository. Replace it
  with your own local path or a public URL if you want another clip.


In [ ]:
RUN_ID = "demo_recruiter_km58_55s"            # new id per run; do not reuse
VIDEO_URL = (
    "https://raw.githubusercontent.com/ibrahim119920/"
    "highway-lane-analytics/master/docs/sample_output/km58_sample_55s.mp4"
)
VIDEO_PATH = Path("data/raw/demo_km58_55s.mp4")
BASE_TIMESTAMP = "2026-11-08T06:58:06"       # overlay time of the sample clip
IMGSZ = 640
FRAME_STRIDE = 1
WRITE_VIDEO = True

# --- upload to the production dashboard ---------------------------------
REPLAY = False                                # True = send to the dashboard
PIPELINE_URL = "https://lane-api.syaifudin-fam.my.id"
PIPELINE_TOKEN = ""                            # owner: paste the token to share uploads
# -------------------------------------------------------------------------

print("run_id      :", RUN_ID)
print("replay      :", REPLAY)
print("device      :", DEVICE)


## 3. Download the sample and run the pipeline

The sample is 55 seconds of real daylight footage from the calibrated camera, so the
shipped lane polygons fit without recalibration. IMGSZ = 640 keeps the CPU demo fast;
set it to 960 to match production.


In [ ]:
import requests

VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)
if not VIDEO_PATH.exists() or VIDEO_PATH.stat().st_size == 0:
    print("downloading", VIDEO_URL)
    response = requests.get(VIDEO_URL, timeout=60)
    response.raise_for_status()
    VIDEO_PATH.write_bytes(response.content)
print("video       :", VIDEO_PATH, f"({VIDEO_PATH.stat().st_size / 1e6:.2f} MB)")

config = load_config("config/config.yaml")
config.video.path = str(VIDEO_PATH)
config.video.frame_stride = FRAME_STRIDE
config.base_timestamp = BASE_TIMESTAMP
config.model.device = DEVICE
config.model.imgsz = IMGSZ
config.evidence.enabled = False      # keep the demo light; the annotated video is enough
config.validate()

batch = process_video(
    config,
    write_video=WRITE_VIDEO,
    generate_reports=True,
    run_id=RUN_ID,
)
print()
print(f"events={batch.total_events} vehicles={batch.total_vehicles} "
      f"hogging={batch.hogging_events} duration={batch.duration_s:.1f}s")
print("reports     :", batch.reports_dir)
print("annotated   :", batch.output_video)


## 4. Watch the annotated result

In [ ]:
from IPython.display import Video, display

if batch.output_video and Path(batch.output_video).exists():
    display(Video(batch.output_video, embed=True, width=900))
else:
    print("No annotated video produced")


## 5. Inspect the numbers

These are the same artifacts the pipeline receives: events, track summaries, lane
stints, and hogging indications.


In [ ]:
import json

import pandas as pd

reports = Path(batch.reports_dir)
summary = json.loads((reports / "summary.json").read_text(encoding="utf-8"))
print("run       :", summary["run"]["run_id"])
print("duration  :", summary["run"]["duration_s"], "s")
print("vehicles  :", summary.get("total_vehicles"))
print("events    :", summary.get("events_by_type"))
print("hogging   :", summary.get("hogging"))

print()
print("Lane usage")
display(pd.read_csv(reports / "lane_usage.csv"))
print("Hogging indications")
hogging_path = reports / "hogging_events.csv"
if hogging_path.exists():
    display(pd.read_csv(hogging_path))


## 6. Optional: send this run to the dashboard

Set `REPLAY = True` in section 2. The sender validates timestamps and camera ids,
waits for the receiver/Kafka, sends events in 5-second windows, tracks once, and
finally `complete=true` so the dashboard builds its aggregates.

> If the dashboard says **"data export is not ready yet"**, the server still needs
> its consumer/export (or `lane-pipeline ingest` / `rebuild-gold`). Ask the operator.


In [ ]:
if REPLAY:
    from lane_analytics.replay import fetch_stats, replay_run, resolve_token

    token = PIPELINE_TOKEN or resolve_token()
    if not token:
        raise SystemExit("REPLAY=True but PIPELINE_TOKEN is empty")

    result = replay_run(
        batch.events_path,
        tracks_path=batch.tracks_path,
        summary_path=str(Path(batch.reports_dir) / "summary.json"),
        config_path=str(Path(batch.reports_dir) / "run_config.json"),
        url=PIPELINE_URL,
        token=token,
        run_id=RUN_ID,
        source="recruiter-demo",
        speed=5.0,
        window_seconds=5.0,
    )
    print(result.as_dict())
    print("stats:", fetch_stats(PIPELINE_URL, token=token))
    print("dashboard: https://lane.syaifudin-fam.my.id")
else:
    print("REPLAY=False - local demo only. Set REPLAY=True to publish this run.")


## 7. Use your own footage

1. Put the clip somewhere readable (for example `/kaggle/input/<dataset>/clip.mp4`).
2. Set `VIDEO_PATH` to that path and `BASE_TIMESTAMP` to the time shown in the video
   overlay (WIB, naive).
3. Change `RUN_ID` so the run is new (reusing an id marks the old run superseded).
4. Re-run sections 3-6.

The lane polygons in `config/config.yaml` are calibrated for KM 58+400 B. For another
camera, run `lane-analytics calibrate` and redraw the polygons first.

**Links:** [dashboard](https://lane.syaifudin-fam.my.id) ·
[repository](https://github.com/ibrahim119920/highway-lane-analytics) ·
[recruiter guide](https://github.com/ibrahim119920/highway-lane-analytics/blob/master/docs/DEMO_FOR_RECRUITERS.md)
